In [ ]:
%pip install -q duckdb numpy numba matplotlib

In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

import time
import numpy as np
import matplotlib.pyplot as plt
from packages.candle_loader import load_candles
from packages.indicators import rolling_robust_z_score, ma, rsi_1_1

In [ ]:
ASSET     = "btcusdt"
DATE_FROM = "2024-01-01 00:00:00"
DATE_TO   = "2026-01-01 00:00:00"

In [ ]:
candles = load_candles(ASSET, DATE_FROM, DATE_TO, columns=["c", "v", "q", "vwap", "vb", "vs"])
# columns: ts=0, c=1, v=2, q=3, vwap=4, vb=5, vs=6

In [ ]:
ZSCORE_WINDOW = 7 * 24 * 60
t0 = time.perf_counter()
v_zscore = rolling_robust_z_score(candles[:, 2], window=ZSCORE_WINDOW)
elapsed = time.perf_counter() - t0
print(f"rolling_robust_z_score elapsed: {elapsed:.4f}s  shape: {v_zscore.shape}")
candles = np.hstack([candles, v_zscore.reshape(-1, 1)])
# column 8 (index 7): rolling_robust_z_score of volume

In [ ]:
MA_WINDOW = 7 * 24 * 60
t0 = time.perf_counter()
v_ma = ma(candles[:, 2], window=MA_WINDOW)
elapsed = time.perf_counter() - t0
print(f"ma elapsed: {elapsed:.4f}s  shape: {v_ma.shape}")
candles = np.hstack([candles, v_ma.reshape(-1, 1)])
# column 9 (index 8): ma of volume

In [ ]:
t0 = time.perf_counter()
rsi_7 = rsi_1_1(candles[:, 4], window=7)
elapsed = time.perf_counter() - t0
print(f"rsi_1_1(window=7) elapsed: {elapsed:.4f}s  shape: {rsi_7.shape}")
candles = np.hstack([candles, rsi_7.reshape(-1, 1)])
# column 10 (index 9): rsi_1_1 of vwap window=7

In [ ]:
t0 = time.perf_counter()
rsi_14 = rsi_1_1(candles[:, 4], window=14)
elapsed = time.perf_counter() - t0
print(f"rsi_1_1(window=14) elapsed: {elapsed:.4f}s  shape: {rsi_14.shape}")
candles = np.hstack([candles, rsi_14.reshape(-1, 1)])
# column 11 (index 10): rsi_1_1 of vwap window=14

In [ ]:
t0 = time.perf_counter()
rsi_60 = rsi_1_1(candles[:, 4], window=60)
elapsed = time.perf_counter() - t0
print(f"rsi_1_1(window=60) elapsed: {elapsed:.4f}s  shape: {rsi_60.shape}")
candles = np.hstack([candles, rsi_60.reshape(-1, 1)])
# column 12 (index 11): rsi_1_1 of vwap window=60

In [ ]:
# Trimming
TRIM = 7 * 24 * 60
before = candles.shape
candles = candles[TRIM:]
print(f"shape before trimming: {before}  after trimming: {candles.shape}")

In [ ]:
LOOK_BACK  = 1440
LOOK_AHEAD = 240
K          = 100

In [ ]:
from packages.asset_analyzer import asset_snapshot_lookback_lookahead_normalize_prepare_all

t0 = time.perf_counter()
asset_snapshot_lookback_lookahead_normalize_prepare_all(candles, LOOK_BACK, LOOK_AHEAD, K)
elapsed = time.perf_counter() - t0
print(f"asset_snapshot_lookback_lookahead_normalize_prepare_all elapsed: {elapsed:.4f}s")